# AlphaCluster — Fine-tune v3 (Phase 3 only, multi-asset)

Continue training the v3 model (500k steps) on **10 crypto assets** using
**Phase 3 only** (low entropy, full penalties). Target: 2x data coverage (~8M steps).

**Why Phase 3 only?** v3 already learned to trade patiently (9 trades/ep).
Restarting curriculum from Phase 1 destroys this behavior.

**Why multi-asset?** Training on 10 pairs prevents overfitting to BTC.
The model sees normalized indicators (returns, RSI, MACD, BB, ATR) — asset-agnostic.

**Phase 3:** ent_coef=0.005, fee=2.0x, churn=2.0x, drawdown=1.5x

**Budget:** ~15h training + eval overhead at ~150 fps. Checkpoint every 1M steps (8 evals).

## Setup

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

!pip install -q "stable-baselines3>=2.4,<3.0" "gymnasium>=1.0,<2.0" pyarrow python-dotenv tqdm rich

import sys
from pathlib import Path

# Add the project source to the Python path
SRC_DIR = "/kaggle/input/alphacluster-repo/src"
sys.path.insert(0, SRC_DIR)

import alphacluster
print(f"AlphaCluster loaded from {Path(alphacluster.__file__).parent}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected.")

## Load & Split Multi-Asset Data

Load all available `*_5m.parquet` files. Each asset is split chronologically
(70/15/15) and the training sets are passed as a list to `TradingEnv(dfs=...)`.

In [ ]:
import pandas as pd

DATA_DIR = Path("/kaggle/input/alphacluster-data")

# Top 10 assets by market cap (balance between diversity and memory)
TOP_10 = [
    "btcusdt", "ethusdt", "bnbusdt", "solusdt", "xrpusdt",
    "dogeusdt", "adausdt", "avaxusdt", "trxusdt", "linkusdt",
]

all_dfs = {}
for symbol in TOP_10:
    path = DATA_DIR / f"{symbol}_5m.parquet"
    if not path.exists():
        print(f"  SKIP {symbol.upper()} (not found)")
        continue
    df = pd.read_parquet(path, engine="pyarrow")
    all_dfs[symbol.upper()] = df
    date_range = f"{df['open_time'].iloc[0]} to {df['open_time'].iloc[-1]}"
    print(f"  {symbol.upper()}: {len(df):>10,} candles  ({date_range})")

print(f"\nLoaded {len(all_dfs)} assets, {sum(len(d) for d in all_dfs.values()):,} total candles")

# Chronological split per asset: 70% train / 15% val / 15% test
train_dfs = []
val_dfs = []
test_dfs = []

for symbol, df in all_dfs.items():
    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)
    train_dfs.append(df.iloc[:train_end].reset_index(drop=True))
    val_dfs.append(df.iloc[train_end:val_end].reset_index(drop=True))
    test_dfs.append(df.iloc[val_end:].reset_index(drop=True))

total_train = sum(len(d) for d in train_dfs)
print(f"Train candles: {total_train:,}")
print(f"Episodes for 1x coverage: ~{total_train // 2016:,}")
print(f"Timesteps for 2x coverage: ~{total_train // 2016 * 2016 * 2:,}")

# BTC splits for evaluation (consistent with v3 baseline)
btc_df = all_dfs["BTCUSDT"]
n_btc = len(btc_df)
btc_val_df = btc_df.iloc[int(n_btc * 0.70):int(n_btc * 0.85)].reset_index(drop=True)
btc_test_df = btc_df.iloc[int(n_btc * 0.85):].reset_index(drop=True)
print(f"BTC eval: val={len(btc_val_df):,}  test={len(btc_test_df):,}")

## Fine-tune v3 (Phase 3 only, 8M steps, 10 assets)

Load the v3 checkpoint and continue training with `phase1_end=0.0, phase2_end=0.0`
(Phase 3 from step 0). Training envs use `dfs=train_dfs` — random asset per episode.

Checkpoint + evaluation on BTC test set every 1M steps. All results saved to CSV
for tracking progression across checkpoints.

In [ ]:
import csv
import time

import torch
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize

from alphacluster.agent.config import TrainingConfig
from alphacluster.agent.trainer import (
    CurriculumCallback,
    create_agent,
    save_agent,
)
from alphacluster.backtest.metrics import calculate_metrics
from alphacluster.backtest.runner import run_backtest
from alphacluster.config import MODEL_VERSION
from alphacluster.env.trading_env import TradingEnv

# ── Paths ──
V3_CHECKPOINT = Path("/kaggle/input/alphacluster-models/ppo_trading_simple_final_v3.pt")
MODELS_DIR = Path("/kaggle/working/models")
CHECKPOINT_DIR = MODELS_DIR / "checkpoints"
RESULTS_CSV = MODELS_DIR / "checkpoint_results.csv"

assert V3_CHECKPOINT.exists(), f"v3 checkpoint not found: {V3_CHECKPOINT}"

FINETUNE_TIMESTEPS = 8_000_000
CHECKPOINT_FREQ = 1_000_000
N_ENVS = 4
EVAL_EPISODES = 5
EVAL_SEED = 42


class ProgressCallback(BaseCallback):
    """Print a one-line progress update every log_interval timesteps."""

    def __init__(self, total_timesteps: int, log_interval: int = 50_000):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self.log_interval = log_interval
        self._start_time = None
        self._next_log = log_interval

    def _on_training_start(self) -> None:
        self._start_time = time.time()

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_log:
            elapsed = time.time() - self._start_time
            pct = self.num_timesteps / self.total_timesteps * 100
            fps = self.num_timesteps / max(elapsed, 1)
            eta = (self.total_timesteps - self.num_timesteps) / max(fps, 1)
            print(
                f"  [{pct:5.1f}%] {self.num_timesteps:>10,} / {self.total_timesteps:,} "
                f"| {fps:.0f} fps | ETA {eta/60:.0f}m",
                flush=True,
            )
            self._next_log += self.log_interval
        return True


class CheckpointEvalCallback(BaseCallback):
    """Save checkpoint + run BTC eval every checkpoint_freq timesteps."""

    def __init__(self, checkpoint_freq, eval_env, models_dir, results_csv,
                 n_episodes=5, seed=42):
        super().__init__(verbose=0)
        self.checkpoint_freq = checkpoint_freq
        self.eval_env = eval_env
        self.models_dir = Path(models_dir)
        self.results_csv = Path(results_csv)
        self.n_episodes = n_episodes
        self.seed = seed
        self._next_checkpoint = checkpoint_freq
        self._results = []

    def _on_step(self) -> bool:
        if self.num_timesteps >= self._next_checkpoint:
            step_m = self.num_timesteps / 1_000_000
            self._next_checkpoint += self.checkpoint_freq

            # Save checkpoint
            ckpt_path = self.models_dir / f"checkpoint_{self.num_timesteps // 1_000_000}M"
            save_agent(self.model, ckpt_path)

            # Evaluate on BTC test set
            result = run_backtest(
                model=self.model, env=self.eval_env,
                n_episodes=self.n_episodes, seed=self.seed,
            )
            metrics = calculate_metrics(result)

            row = {
                "timesteps": self.num_timesteps,
                "step_M": f"{step_m:.0f}M",
                "avg_pnl_pct": round(metrics["avg_episode_return_pct"], 2),
                "trades_per_ep": round(metrics["avg_trades_per_episode"], 1),
                "win_rate": round(metrics["win_rate_pct"], 1),
                "profit_factor": round(metrics.get("profit_factor", 0), 4),
                "avg_trade_duration": round(metrics.get("avg_trade_duration", 0), 1),
                "max_drawdown_pct": round(metrics.get("max_drawdown_pct", 0), 2),
                "sharpe": round(metrics.get("sharpe_ratio", 0), 4),
            }
            self._results.append(row)

            # Write CSV (overwrite each time for safety)
            self.results_csv.parent.mkdir(parents=True, exist_ok=True)
            with open(self.results_csv, "w", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=row.keys())
                writer.writeheader()
                writer.writerows(self._results)

            print(f"\n  >>> CHECKPOINT {row['step_M']}: "
                  f"PnL={row['avg_pnl_pct']}%, "
                  f"trades/ep={row['trades_per_ep']}, "
                  f"WR={row['win_rate']}%, "
                  f"PF={row['profit_factor']}, "
                  f"Sharpe={row['sharpe']}\n")

        return True

    @property
    def results(self):
        return self._results


def make_env(dfs_list, config, rank=0):
    """Factory for SubprocVecEnv with multi-asset support."""
    def _init():
        import os
        import warnings

        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        warnings.filterwarnings("ignore", message=".*Gym has been unmaintained.*")
        warnings.filterwarnings("ignore", message=".*datetime.datetime.utcnow.*")

        env = TradingEnv(
            dfs=dfs_list,
            window_size=config.window_size,
            episode_length=config.episode_length,
            simple_actions=True,
            fixed_size_pct=config.fixed_size_pct,
            fixed_leverage=config.fixed_leverage,
        )
        env.reset(seed=rank)
        return env
    return _init


# ── Config: Phase 3 only ──
config = TrainingConfig(
    total_timesteps=FINETUNE_TIMESTEPS,
    simple_actions=True,
    fixed_size_pct=0.10,
    fixed_leverage=10,
    eval_freq=CHECKPOINT_FREQ,
    phase1_end=0.0,
    phase2_end=0.0,
)

# ── Environments ──
print(f"Creating {N_ENVS} parallel training environments ({len(train_dfs)} assets) ...")
envs = SubprocVecEnv([make_env(train_dfs, config, rank=i) for i in range(N_ENVS)])
train_env = VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env: BTC test set (for checkpoint evaluation)
eval_env = TradingEnv(
    df=btc_test_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=True,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

# ── Load v3 weights ──
print("Creating PPO agent and loading v3 checkpoint ...")
agent = create_agent(train_env, config, verbose=0)
state_dict = torch.load(str(V3_CHECKPOINT), map_location="cpu", weights_only=True)
agent.policy.load_state_dict(state_dict)
print(f"Loaded v3 weights from {V3_CHECKPOINT}")

# ── Callbacks ──
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
curriculum_cb = CurriculumCallback(config, verbose=1)
progress_cb = ProgressCallback(FINETUNE_TIMESTEPS, log_interval=100_000)
checkpoint_cb = CheckpointEvalCallback(
    checkpoint_freq=CHECKPOINT_FREQ,
    eval_env=eval_env,
    models_dir=CHECKPOINT_DIR,
    results_csv=RESULTS_CSV,
    n_episodes=EVAL_EPISODES,
    seed=EVAL_SEED,
)

# ── Train ──
print(f"\nFine-tuning v3 for {FINETUNE_TIMESTEPS:,} steps (Phase 3 only, multi-asset)")
print(f"Model version: {MODEL_VERSION}")
print(f"Phase 3: ent_coef=0.005, fee=2.0x, churn=2.0x, drawdown=1.5x")
print(f"Config: size_pct={config.fixed_size_pct}, leverage={config.fixed_leverage}x")
print(f"Assets: {len(train_dfs)}, checkpoint+eval every {CHECKPOINT_FREQ//1_000_000}M steps")
print(f"Estimated time: ~{FINETUNE_TIMESTEPS / 150 / 3600:.0f}h at 150 fps\n")

start = time.time()
agent.learn(
    total_timesteps=FINETUNE_TIMESTEPS,
    callback=CallbackList([curriculum_cb, progress_cb, checkpoint_cb]),
    progress_bar=False,
)
elapsed = time.time() - start
print(f"\nTraining complete in {elapsed/60:.1f} minutes ({elapsed/3600:.1f}h)")

# ── Save final model ──
MODELS_DIR.mkdir(parents=True, exist_ok=True)
final_path = MODELS_DIR / "ppo_trading_simple_v3_finetuned"
save_agent(agent, final_path)
print(f"Final model saved to {final_path}.pt")

train_env.close()

## Final Evaluation + Best Checkpoint Selection

Run full eval on BTC test set for the final model, then find the best checkpoint.

In [ ]:
from alphacluster.backtest.runner import run_backtest
from alphacluster.backtest.metrics import calculate_metrics, print_report

# ── Final model eval ──
test_env = TradingEnv(
    df=btc_test_df,
    window_size=config.window_size,
    episode_length=config.episode_length,
    simple_actions=True,
    fixed_size_pct=config.fixed_size_pct,
    fixed_leverage=config.fixed_leverage,
)

print("Final model evaluation on BTC TEST set (5 episodes, seed=42):\n")
result = run_backtest(model=agent, env=test_env, n_episodes=EVAL_EPISODES, seed=EVAL_SEED)
metrics = calculate_metrics(result)
print_report(metrics)

# ── Checkpoint progression ──
print("\n" + "=" * 80)
print("  CHECKPOINT PROGRESSION (BTC test set)")
print("=" * 80)

results_df = pd.read_csv(RESULTS_CSV)
print(results_df.to_string(index=False))

# ── Best checkpoint ──
best_idx = results_df["avg_pnl_pct"].idxmax()
best = results_df.iloc[best_idx]
print(f"\n  BEST CHECKPOINT: {best['step_M']} — "
      f"PnL={best['avg_pnl_pct']}%, WR={best['win_rate']}%, PF={best['profit_factor']}")

# ── v3 comparison ──
print(f"\n  v3 baseline:     PnL=-0.95%, trades/ep=8.8, WR=40.9%, PF=0.886")
print(f"  Best checkpoint: PnL={best['avg_pnl_pct']}%, "
      f"trades/ep={best['trades_per_ep']}, "
      f"WR={best['win_rate']}%, PF={best['profit_factor']}")

improved = best["avg_pnl_pct"] > -0.95
print(f"\n  {'IMPROVED' if improved else 'NO IMPROVEMENT'} over v3 baseline")
print("=" * 80)

## Training Progression Charts

In [ ]:
import matplotlib.pyplot as plt

if RESULTS_CSV.exists():
    df_r = pd.read_csv(RESULTS_CSV)
    steps = df_r["timesteps"] / 1_000_000

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle("v3 Fine-tuning Progression (Phase 3, multi-asset)", fontsize=14)

    plots = [
        ("avg_pnl_pct", "Avg PnL per Episode (%)"),
        ("trades_per_ep", "Trades per Episode"),
        ("win_rate", "Win Rate (%)"),
        ("profit_factor", "Profit Factor"),
        ("avg_trade_duration", "Avg Trade Duration (steps)"),
        ("sharpe", "Sharpe Ratio"),
    ]

    for ax, (col, title) in zip(axes.flat, plots):
        if col in df_r.columns:
            ax.plot(steps, df_r[col], marker="o", markersize=5, linewidth=2)
            # v3 baseline reference lines
            baselines = {
                "avg_pnl_pct": -0.95,
                "trades_per_ep": 8.8,
                "win_rate": 40.9,
                "profit_factor": 0.886,
                "avg_trade_duration": 228,
                "sharpe": -1.29,
            }
            if col in baselines:
                ax.axhline(y=baselines[col], color="red", linestyle="--",
                           alpha=0.6, label="v3 baseline")
                ax.legend(fontsize=8)
        ax.set_title(title)
        ax.set_xlabel("Steps (M)")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No checkpoint results found. Run training first.")

## Download

In [ ]:
import os

print("=" * 60)
print("  FINE-TUNING COMPLETE")
print("=" * 60)
print()
print("Model files in /kaggle/working/ (download from Output tab):")
print()

for root, dirs, files in os.walk("/kaggle/working/models"):
    level = root.replace("/kaggle/working/", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = "  " * (level + 1)
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(root, f)) / (1024 * 1024)
        print(f"{sub_indent}{f}  ({size_mb:.1f} MB)")

print()
print("To evaluate locally:")
print("  python scripts/evaluate.py --model-path models/ppo_trading_simple_v3_finetuned.pt --simple-actions")